# **1. Import required Python packages**

This cell loads a bunch of Python packages which are required to run the notebook. You should use the **cheminf.yaml** environment for running this notebook!

For example, some of these are related to the processing of data (`numpy` and `pandas`), some for actually performing the linear modelling (`sklearn`) and some for plotting the results of your modeling (`matplotlib` and `seaborn`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
sns.set_style("white")

from sklearn import tree
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier


# **2. Load in your data**

This cell loads the data in your csv file to a pandas dataframe.

In [ ]:
df = pd.read_csv("data/homo_diels_alder.csv", index_col=0)

Check to make sure the dataframe looks ok:

In [ ]:
df.head()

df.dropna(subset=["mvk_hda_yield"], inplace=True)

df

It can also be useful to generate a dataframe with only the features when we model with them later.

In [ ]:
df_features = df.copy().drop(columns=["mvk_hda_yield"])
df_features.head()

# **3. Examine the data spread**

Next, we will plot a histogram to visually represent the spread of data (the spread of our `response_col`).

In [ ]:
response_col = "mvk_hda_yield"

sns.histplot(data=df, x=response_col)
plt.show()

# **4. Run the classification model (single node decision tree)**

Look at the histogram - this usually gives us a good idea where to apply the `y_cut`. Define the value of your `y_cut` below:

In [ ]:
y_cut = 20

Before we can run our classification model, we should prepare our data into `numpy` arrays to give to the algorithms within `sklearn`:

In [ ]:
X = df_features.to_numpy()
X_names = df_features.columns
y = df[response_col].to_numpy()

All this runs the single-node decision tree classification (using the `DecisionTreeClassifer` in scikit-learn, note the `max_depth=1`). It will also print a nice visualization of the classification, the statistics, a depiction of the actual decision tree and the confusion matrix.

In [ ]:
features = range(np.shape(X)[1])
y_class = np.array([0 if i < y_cut else 1 for i in y])
n_classes = 2

# Setup decision tree classifier
dt = DecisionTreeClassifier(max_depth=1).fit(X, y_class)

feat = int(np.argmax(dt.feature_importances_))

dt_plt = DecisionTreeClassifier(max_depth=1).fit(X[:,feat].reshape(-1, 1), y_class)

print("Classification analysis:\n------------------------------")
print(f"Number of samples: {len(y_class)}")
print(f"Feature: {X_names[feat]}")
print(f"Decision threshold: {dt_plt.tree_.threshold[0]:.2f}")
print(f"Accuracy: {dt_plt.score(X[:,feat].reshape(-1, 1), y_class):.2f}")
print(f"Precision: {metrics.precision_score(y_class,dt_plt.predict(X[:,feat].reshape(-1, 1))):.2f}")
print(f"Recall: {metrics.recall_score(y_class,dt_plt.predict(X[:,feat].reshape(-1, 1))):.2f}")
print(f"F1 Score: {metrics.f1_score(y_class,dt_plt.predict(X[:,feat].reshape(-1, 1))):.2f}")

plot_colors = "rb"

plt.figure(figsize=(6, 6))    
plt.xlabel(X_names[feat])
plt.ylabel(response_col)

for i, color in zip(range(n_classes), plot_colors):
    idx = np.where(y_class == i)
    plt.scatter(X[idx, feat], y[idx], c=color,cmap=plt.cm.RdBu, edgecolor='black', s=50)    

plt.axhline(y=y_cut, color='k', linestyle='--',linewidth=2)
plt.axvline(x=dt_plt.tree_.threshold[0], color='k', linestyle='--',linewidth=2)
plt.show()

# plot tree
tree.plot_tree(dt_plt, feature_names=[X_names[feat]], class_names=[f"{response_col}<{y_cut}", f"{response_col}>={y_cut}"], filled=True)
plt.show()

# plot confusion matrix
cm = metrics.confusion_matrix(y_class, dt_plt.predict(X[:, feat].reshape(-1, 1)))
disp = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[f"{response_col}<{y_cut}", f"{response_col}>={y_cut}"],
)
disp.plot(colorbar=False)
plt.show()
